## Allocate GPU memory manually

- `torch.empty()` creates a tensor without initializing its values. It allocates memory for the tensor, but `PyTorch does not fill that memory with zeros or any default value`. Whatever data was already present in that memory location remains there. Just Allocate memory & Do nothing else.
    ```table
    Allocate tensor (empty)
    ↓
    Initialize weights
    ↓
    Train model
    ```

- `torch.zeros((2,3))` Allocate memory Fill every byte with 0.
    ```table
    torch.zeros(...)
    ↓
    Overwrite everything 
    ```
    | Function | Allocates Memory | Initializes Values |
    | :--- | :--- | :--- |
    | `torch.empty()` | ✅ | ❌ |
    | `torch.zeros()` | ✅ | ✅ (0) |
    | `torch.ones()` | ✅ | ✅ (1) |
    | `torch.full()` | ✅ | ✅ (specified value) |
    | `torch.randn()` | ✅ | ✅ (random normal) |

- Q. ``torch.empty()`` default float32 how to set for float16 int 32 etc... ?
```python
x = torch.empty((2, 3), dtype=torch.float16)
torch.float16     # Half precision
torch.bfloat16    # Brain floating point
torch.float32     # Single precision (default)
torch.float64     # Double precision
```


#### Monitor GPU memory allocation

In [1]:
def show_gpu_memory():
    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2

    print(f"Allocated: {allocated:.2f} MiB")
    print(f"Reserved:  {reserved:.2f} MiB")

In [2]:
import torch

device = "cuda"

# Allocate GPU memory (uninitialized)
x = torch.empty(
    (4096, 4096),
    dtype=torch.float32,
    device=device
)

# Initialize memory
x.fill_(1.0)

print("Device:", x.device)
print("Element size",x.element_size(),": Memory:", x.numel() * x.element_size() / 1024**2, "MiB")
show_gpu_memory()

# Release tensor reference
del x

print("\nAfter del:")
show_gpu_memory()

# Release unused cached memory
torch.cuda.empty_cache()

print("\nAfter empty_cache:")
show_gpu_memory()

Device: cuda:0
Element size 4 : Memory: 64.0 MiB
Allocated: 64.00 MiB
Reserved:  64.00 MiB

After del:
Allocated: 0.00 MiB
Reserved:  64.00 MiB

After empty_cache:
Allocated: 0.00 MiB
Reserved:  0.00 MiB


## Allocate and release memory during model training

In [3]:
import torch
import torch.nn as nn

device = "cuda"

model = nn.Linear(4096, 4096).to(device)

optimizer = torch.optim.AdamW(model.parameters())
total_params = sum(
    p.numel() for p in model.parameters()
)

print("Total parameters:", total_params)
print("Weight shape:", model.weight.shape)
print("Weight parameters:", model.weight.numel())

print("Bias shape:", model.bias.shape)
print("Bias parameters:", model.bias.numel())

for step in range(5):

    # Allocate input GPU memory
    x = torch.randn(32, 4096, device=device) # First dimension (batch size): x.shape[0] or x.size(0) → 32
                                             # Second dimension (features): x.shape[1] or x.size(1) → 4096
    print('x shape: ',x.shape)
    print('x size: ',x.size())
    show_gpu_memory()
    optimizer.zero_grad(set_to_none=True)

    # Forward
    output = model(x)
    print('output shape: ',output.shape)
    print('output size: ',output.size())

    # Loss
    loss = output.mean()

    # Backward
    loss.backward()

    # Update parameters
    optimizer.step()

    # Release temporary references
    del x, output, loss

    print(f"Step {step}:")
    show_gpu_memory()

Total parameters: 16781312
Weight shape: torch.Size([4096, 4096])
Weight parameters: 16777216
Bias shape: torch.Size([4096])
Bias parameters: 4096
x shape:  torch.Size([32, 4096])
x size:  torch.Size([32, 4096])
Allocated: 64.52 MiB
Reserved:  66.00 MiB
output shape:  torch.Size([32, 4096])
output size:  torch.Size([32, 4096])
Step 0:
Allocated: 320.06 MiB
Reserved:  386.00 MiB
x shape:  torch.Size([32, 4096])
x size:  torch.Size([32, 4096])
Allocated: 320.56 MiB
Reserved:  386.00 MiB
output shape:  torch.Size([32, 4096])
output size:  torch.Size([32, 4096])
Step 1:
Allocated: 320.06 MiB
Reserved:  386.00 MiB
x shape:  torch.Size([32, 4096])
x size:  torch.Size([32, 4096])
Allocated: 320.56 MiB
Reserved:  386.00 MiB
output shape:  torch.Size([32, 4096])
output size:  torch.Size([32, 4096])
Step 2:
Allocated: 320.06 MiB
Reserved:  386.00 MiB
x shape:  torch.Size([32, 4096])
x size:  torch.Size([32, 4096])
Allocated: 320.56 MiB
Reserved:  386.00 MiB
output shape:  torch.Size([32, 4096])


In above example GPU memory is consumed by model parameters, input tensors, activations, gradients and optimizer states.

In [4]:
import torch
import torch.nn as nn

device = "cuda"

def show_memory(label):

    torch.cuda.synchronize()

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    peak = torch.cuda.max_memory_allocated() / 1024**2

    print(f"\n{label}")
    print(f"Allocated: {allocated:.2f} MiB")
    print(f"Reserved : {reserved:.2f} MiB")
    print(f"Peak     : {peak:.2f} MiB")


# Reset peak statistics
torch.cuda.reset_peak_memory_stats()

show_memory("1. Before model")

# Create model
model = nn.Linear(4096, 4096).to(device)

show_memory("2. After model allocation")

# Create optimizer
optimizer = torch.optim.AdamW(model.parameters())

# Create input
x = torch.randn(32, 4096, device=device)

show_memory("3. After input allocation")

# Forward
output = model(x)

show_memory("4. After forward")

# Loss
loss = output.mean()

# Backward
loss.backward()

show_memory("5. After backward")

# Optimizer update
optimizer.step()

show_memory("6. After optimizer step")

# Release temporary tensors
del x, output, loss

torch.cuda.empty_cache()

show_memory("7. After releasing temporary tensors")


1. Before model
Allocated: 320.06 MiB
Reserved : 386.00 MiB
Peak     : 320.06 MiB

2. After model allocation
Allocated: 384.08 MiB
Reserved : 386.00 MiB
Peak     : 384.08 MiB

3. After input allocation
Allocated: 128.52 MiB
Reserved : 386.00 MiB
Peak     : 384.08 MiB

4. After forward
Allocated: 129.02 MiB
Reserved : 386.00 MiB
Peak     : 384.08 MiB

5. After backward
Allocated: 193.03 MiB
Reserved : 386.00 MiB
Peak     : 384.08 MiB

6. After optimizer step
Allocated: 321.06 MiB
Reserved : 386.00 MiB
Peak     : 385.08 MiB

7. After releasing temporary tensors
Allocated: 320.06 MiB
Reserved : 322.00 MiB
Peak     : 385.08 MiB


## Calculate actual GPU memory using PyTorch

#### Calculate GPU memory for full fine-tuning
During full fine-tuning, all model parameters are trainable.

- We need memory for four major components:
    - Model weights
    - Gradients
    - Optimizer states
    - Activations

For a Linear layer, always remember:

$$\boxed{Parameters = (Input \times Output) + Output}$$

This formula is useful when calculating model size and GPU memory requirements for LLMs.


**1. Understand Linear layer parameters**

PyTorch Linear layer uses:

$$Y = XW^T + b$$

In above code Where:
* Input features = 4096
* Output features = 4096
* Weight matrix = 4096 × 4096
* Bias vector = 4096

**Step 1. Understand the weight matrix**

For `nn.Linear(in_features, out_features)`:

$$Weight\ Shape = (out\_features,\ in\_features)$$

Therefore:

$$Weight\ Shape = (4096,\ 4096)$$

**Step 2. Calculate weight parameters**

A matrix contains rows × columns elements.

$$4096 \times 4096$$

$$= 16,777,216$$

> **16,777,216 Weight parameters**

**Step 3. Calculate bias parameters**

Bias contains one parameter per output feature.

$$Bias = 4096$$

**Step 4. Calculate total parameters**

$$Total = Weight + Bias$$

$$= 16,777,216 + 4096$$

$$= 16,781,312$$



#### Calculate model parameter memory
```python
model = nn.Linear(4096, 4096)
```
A Linear layer contains:
* Weight matrix: 4096 × 4096
* Bias: 4096

Total parameters:

$$P = (4096 \times 4096) + 4096$$

$$P = 16,781,312$$

Each FP32 parameter consumes 4 bytes.

$$Memory = P \times 4$$

#### Calculate input tensor memory
```python
x = torch.randn(32, 4096, device="cuda")
```
Number of elements:

$$32 \times 4096 = 131072$$

FP32 requires 4 bytes per element.

$$Memory = 131072 \times 4$$

#### Calculate output tensor memory
```python
output = model(x)
```
Output shape:

$$32 \times 4096$$

Memory:

$$32 \times 4096 \times 4$$

#### Calculate gradient memory
```python
loss.backward()
```
PyTorch calculates gradients for trainable model parameters.

The model has 16,781,312 parameters.

Assuming FP32 gradients:

$$Gradient\ Memory = P \times 4$$

#### Calculate AdamW optimizer memory

AdamW maintains two additional state tensors for each trainable parameter:
* First moment (`exp_avg`)
* Second moment (`exp_avg_sq`)

Assuming FP32 optimizer states:

$$Optimizer\ Memory = P \times 4 \times 2$$

> **128.06 MiB**

These optimizer states are generally initialized when the optimizer first performs an update.

#### Total GPU memory calculation

Let's combine the components.

| Component | GPU memory |
| :--- | ---: |
| Model parameters | 64.03 MiB |
| Input tensor | 0.50 MiB |
| Output tensor | 0.50 MiB |
| Gradients | 64.03 MiB |
| AdamW optimizer states | 128.06 MiB |
| **Total** | **257.12 MiB** |



## Gradient Accumulation in PyTorch, How to reduce GPU memory consumption?

**Gradient Accumulation** is a technique used to train large Deep Learning and LLM models when GPU memory is limited.

Instead of processing a large batch of training samples in one forward and backward pass, we divide it into smaller batches (micro-batches), calculate gradients for each micro-batch, accumulate those gradients, and update model weights only after processing all micro-batches.

**Main objective:** Reduce GPU memory consumption while maintaining a larger effective batch size.

**1. Why do we need Gradient Accumulation?**

Suppose you are fine-tuning a 7B LLM using PyTorch.

Your GPU memory is limited, and your model cannot process a batch of 32 samples simultaneously.

For example:
* Desired batch size = 32
* GPU can process only 8 samples at a time.
* You cannot fit all 32 samples into GPU memory.

Instead, divide the batch into four micro-batches.

**3. Mathematical explanation**

Suppose we have four micro-batches.

Their individual gradients are:

$$g_1, \quad g_2, \quad g_3, \quad g_4$$

The accumulated gradient is:

$$G = g_1 + g_2 + g_3 + g_4$$

For an equivalent average gradient across equally sized micro-batches, divide each micro-batch loss by 4 before calling `backward()`:

$$G_{\text{avg}} = \frac{g_1 + g_2 + g_3 + g_4}{4}$$

This is why we divide the loss by `accumulation_steps`.

#### Without Gradient Accumulation

```text
+----------------------------------------------------+

|                  Batch size = 32                   |
|                     32 samples                     |
|         All samples processed simultaneously       |
+----------------------------------------------------+
```
> **Higher activation memory**

#### With Gradient Accumulation

```text
+-------------------------+-------------------------+

|      Micro-batch 1      |      Micro-batch 2      |
|        8 Samples        |        8 Samples        |
+-------------------------+-------------------------+

|      Micro-batch 3      |      Micro-batch 4      |
|        8 Samples        |        8 Samples        |
+-------------------------+-------------------------+
                          │
                          ▼
+---------------------------------------------------+

|                Accumulate gradients               |
|               Four backward passes                |
+---------------------------------------------------+
                          │
                          ▼
+---------------------------------------------------+

|               One optimizer update                |
|            After processing 32 samples            |
+---------------------------------------------------+
```
> **Lower activation memory per pass**

**Step 1: First micro-batch**

```python
loss = model(batch_1)
loss.backward()
```

PyTorch calculates gradients and stores them in parameter `.grad` tensors.

**Step 2: Second micro-batch**

```python
loss = model(batch_2)
loss.backward()
```

New gradients are added to existing gradients.

**Step 3: Third micro-batch**

```python
loss = model(batch_3)
loss.backward()
```
**Step 4: Fourth micro-batch**

```python
loss = model(batch_4)
loss.backward()
```

Now gradients from all four micro-batches have accumulated.

**Step 5: Update model weights**

```python
optimizer.step()
optimizer.zero_grad()
```

The optimizer updates model parameters once using the accumulated gradients.

The important point is that `backward()` accumulates gradients by default; it does not automatically clear previous gradients.

#### How to Decided Micro batch Size & Accumulations Steps

$$\text{Effective Batch Size} = \text{Micro Batch Size} \times \text{Accumulation Steps}$$

- For example, if your target batch size is 32:

    | Micro-batch size | Accumulation steps | Effective batch size |
    | ---: | ---: | ---: |
    | 32 | 1 | 32 |
    | 16 | 2 | 32 |
    | 8 | 4 | 32 |
    | 4 | 8 | 32 |
    | 2 | 16 | 32 |

- Start with micro batch size, follow below rule
    * Start with micro_batch_size = 1.
    * Run a forward and backward pass.
    * Check peak GPU memory.
    * Increase to 2, 4, 8, and so on.
    * If you encounter CUDA Out of Memory (OOM), reduce the batch size.


## Peak Memory CUDA

Peak Memory = the maximum amount of GPU memory allocated at any moment during program execution. CUDA benchmarking and LLM training, peak memory is more important than current memory, because it determines whether your model can run without running out of GPU memory.

- Simple Analogy

    - Think of GPU memory like a water tank:
        - Current Memory = water currently in the tank.
        - Peak Memory = highest water level ever reached.
        - reset_peak_memory_stats() = resetting the "highest water level" marker.
        - OOM Error = water level exceeds tank capacity.

#### Q. I have two memory allocated method, memory allocate & another peak allocate. Explain when we choice which memory I need to allocate ?
- 
    | Function | Meaning | When to Use |
    | :--- | :--- | :--- |
    | `torch.cuda.memory_allocated()` | Memory currently occupied by tensors | Check current GPU usage |
    | `torch.cuda.max_memory_allocated()` | Maximum memory occupied at any point since last reset | Measure GPU requirement of an operation |

- Current Memory Allocation: 
    - torch.cuda.memory_allocated()
    - "How much GPU memory am I using right now?"
    - You would use this when:
        - Monitoring GPU usage
        - Debugging memory leaks
        - Checking memory before/after deleting tensors
- Peak Memory Allocation:
    - torch.cuda.max_memory_allocated()
    - "What was the highest GPU memory usage reached?"

#### For ML Engineers:
During benchmarking, almost always use:
```python
torch.cuda.reset_peak_memory_stats()

# inference or training

peak_gb = torch.cuda.max_memory_allocated() / 1024**3
print(f"Peak GPU Memory: {peak_gb:.2f} GB")
```
because **peak memory determines whether a model can fit on a GPU**, while **current memory is only a snapshot at one instant**.


## GPU Out-of-Memory (OOM)

GPU Out-of-Memory (OOM) occurs when a program requests more GPU memory than is currently available for an operation.

In PyTorch, you typically see an error like:
```text
torch.cuda.OutOfMemoryError:
CUDA out of memory. Tried to allocate 2.00 GiB.
GPU 0 has 24.00 GiB total capacity;
20.00 GiB already allocated.
```
This means PyTorch could not allocate the additional memory required to continue the operation.

During model training, GPU memory is consumed by several components:

| Component | Purpose |
| :--- | :--- |
| Model weights | Store model parameters |
| Activations | Store intermediate forward-pass results for backpropagation |
| Gradients | Store calculated gradients |
| Optimizer states | Store Adam momentum and variance |
| Temporary buffers | Store intermediate CUDA computation results |

For example, when training a 7B parameter model using full fine-tuning, weights, gradients, optimizer states, and activations can consume substantial GPU memory.

#### Common causes of GPU OOM:
**1. Large batch size**

A larger batch requires more activation memory.

```python
batch_size = 64 # Higher memory consumption
```

Try reducing it:

```python
batch_size = 8
```

**2. Large model size**

A model with more parameters requires more memory for weights, gradients, and optimizer states.

**3. Large sequence length**

In Transformer models, longer input sequences increase activation memory and attention computation requirements.

For example:

```python
max_length = 4096
```

Reducing it to 2048 may reduce memory consumption.

**4. Gradient accumulation configured incorrectly**

Gradient accumulation does not automatically reduce memory. You must reduce the micro-batch size to reduce activation memory.

**5. Memory fragmentation or cached memory**

PyTorch uses a caching allocator. Sometimes memory is reserved but not immediately available in the way a particular allocation requires. Fragmentation can contribute to allocation failures.

#### How to debug GPU OOM in PyTorch?

Check GPU memory using:

```python
import torch

print("Allocated:",
      torch.cuda.memory_allocated() / 1024**3, "GiB")

print("Reserved:",
      torch.cuda.memory_reserved() / 1024**3, "GiB")

print("Peak:",
      torch.cuda.max_memory_allocated() / 1024**3, "GiB")
```

You can also use Linux:

```bash
nvidia-smi
```
#### How to fix GPU OOM?

| Technique | How it helps |
| :--- | :--- |
| Reduce batch size | Reduces activation memory |
| Gradient accumulation | Allows smaller micro-batches while maintaining a larger effective batch |
| Mixed precision (FP16/BF16) | Reduces memory used by many model tensors and activations |
| Gradient checkpointing | Recomputes activations during backward instead of storing all of them |
| LoRA | Trains fewer parameters and reduces gradient/optimizer memory |
| QLoRA | Quantizes the base model to reduce weight memory |
| Reduce sequence length | Reduces activation and attention memory |
| `optimizer.zero_grad(set_to_none=True)` | Avoids retaining zero-filled gradient tensors between updates |
| `torch.cuda.empty_cache()` | Releases unused cached memory back to the CUDA allocator |

**Question:** Your PyTorch model runs successfully with batch size 8 but gives CUDA OOM with batch size 32. How would you solve it?

**Answer:**

1. Check allocated and reserved GPU memory.
2. Reduce the micro-batch size from 32 to 8.
3. Use gradient accumulation to achieve the desired effective batch size.
4. Enable BF16/FP16 mixed precision.
5. Enable gradient checkpointing if activation memory is high.
6. If fine-tuning an LLM, consider LoRA or QLoRA.
7. Profile memory usage to identify the largest consumers.


#### what is good practices on tanning LLM,  Peak memory allocations or run time memory allocations?

### measure GPU memory in PyTorch

When training LLMs, monitor both current and peak GPU memory. Peak memory is important for selecting safe batch sizes and avoiding OOM during forward, backward and optimizer operations. Runtime memory helps identify memory growth, leaks and unexpected allocation patterns. I also monitor reserved memory and use PyTorch memory profiling to identify bottlenecks.

Following this order:

| Priority | Practice | Purpose |
| ---: | :--- | :--- |
| 1 | Measure peak GPU memory | Establish memory requirements |
| 2 | Start with a small micro-batch | Avoid initial OOM |
| 3 | Use BF16 mixed precision | Reduce tensor memory |
| 4 | Enable gradient checkpointing | Reduce activation memory |
| 5 | Use gradient accumulation | Achieve larger effective batch size |
| 6 | Use LoRA/QLoRA | Reduce fine-tuning memory requirements |
| 7 | Optimize sequence length | Reduce activation and attention memory |
| 8 | Profile training iterations | Identify memory bottlenecks |


```python
# Run training
output = model(inputs)
loss = criterion(output, targets)
loss.backward()
optimizer.step()

# Current memory allocation
current = torch.cuda.memory_allocated() / 1024**3
# Peak memory allocation
peak = torch.cuda.max_memory_allocated() / 1024**3
# Current reserved memory
reserved = torch.cuda.memory_reserved() / 1024**3
# Peak reserved memory
peak_reserved = torch.cuda.max_memory_reserved() / 1024**3
print(f"Current allocated: {current:.2f} GiB")
print(f"Peak allocated: {peak:.2f} GiB")
print(f"Current reserved: {reserved:.2f} GiB")
print(f"Peak reserved: {peak_reserved:.2f} GiB")
```